# Day 9 — Silver: Invoice Metadata (Bronze PDF paths → Silver Delta)

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/invoices/YYYY/MM/DD/*.pdf`

**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/invoices/` (Delta)

**What Bronze has:** Raw PDF files under `bronze-volume/invoices/YYYY/MM/DD/`.
No `_metadata` Delta table is written by Bronze — metadata is extracted here from file paths.

**What Silver adds:**
- Metadata extracted from PDF path: `invoices/YYYY/MM/DD/INV-AU-YYYY-NNNN.pdf`
- Proper typed columns (invoice_date as date, file_size_kb as decimal)
- `invoice_number` parsed from invoice_id (trailing numeric part)
- `invoice_year/month/day` as integers
- Silver audit columns
- Deduplicated on `invoice_id`

**Note:** PDF content parsing (amounts, line items) is a Gold-layer concern.

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Read parameters from Databricks Job ONLY

def _get_job_param(key):
    try:
        value = dbutils.widgets.get(key).strip()
    except Exception:
        raise ValueError(f"Parameter '{key}' was not provided. Run via Databricks Job only.")
    if not value:
        raise ValueError(f"Parameter '{key}' is empty.")
    return value

LOAD_TYPE       = _get_job_param("load_type")
INGESTION_YEAR  = _get_job_param("ingestion_year")
INGESTION_MONTH = _get_job_param("ingestion_month")

# Day is optional — blank = process all days in the month
try:
    INGESTION_DAY = dbutils.widgets.get("ingestion_day").strip()
except Exception:
    INGESTION_DAY = ""

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f"load_type       : {LOAD_TYPE}")
print(f"ingestion_year  : {INGESTION_YEAR}")
print(f"ingestion_month : {INGESTION_MONTH}")
print(f"ingestion_day   : {INGESTION_DAY or '(all days)'}")
print("Parameters validated.")

In [ ]:
# Cell 3 — Constants
BRONZE_INVOICES      = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/invoices"
SILVER_PATH          = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/invoices"
QUARANTINE_PATH      = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/quarantine/invoices"

PIPELINE_NAME = "job_silver_blob_invoices_v1"
RUN_TS        = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze invoices : {BRONZE_INVOICES}")
print(f"Silver path     : {SILVER_PATH}")
print(f"RUN_TS          : {RUN_TS}")

In [ ]:
# Cell 4 — Scan Bronze PDF files and build metadata rows
# Bronze layout: invoices/YYYY/MM/DD/INV-AU-YYYY-NNNNNNNN.pdf
# Bronze stores raw PDFs only — no _metadata Delta table exists.
# Metadata is derived here from file path + file size.

import re
from pyspark.sql import Row

def list_pdfs(path):
    results = []
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return results
    for item in items:
        if item.isDir():
            results.extend(list_pdfs(item.path))
        elif item.name.endswith(".pdf"):
            results.append(item)
    return results

def folder_exists_dbfs(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

# Set scan path based on load_type
if LOAD_TYPE == "incremental":
    scan_path = f"{BRONZE_INVOICES}/{INGESTION_YEAR}/{INGESTION_MONTH}"
    if INGESTION_DAY:
        scan_path = f"{scan_path}/{INGESTION_DAY}"
    print(f"Mode: incremental — scanning {scan_path}")
else:
    scan_path = BRONZE_INVOICES
    print(f"Mode: full — scanning {scan_path}")

if not folder_exists_dbfs(scan_path):
    raise Exception(
        f"Bronze path not found: {scan_path}\n"
        f"Run day_6/04_bronze_blob_invoices_pdf.ipynb first."
    )

pdf_files = list_pdfs(scan_path)
print(f"PDF files found: {len(pdf_files)}")

rows = []
for f in pdf_files:
    m = re.search(r"invoices/(\d{4})/(\d{2})/(\d{2})/(.+\.pdf)$", f.path)
    if m:
        rows.append(Row(
            invoice_id   = m.group(4).replace(".pdf", ""),
            year         = m.group(1),
            month        = m.group(2),
            day          = m.group(3),
            file_size_kb = round(f.size / 1024, 1),
            bronze_path  = f.path,
        ))

bronze_count = len(rows)
print(f"Metadata rows extracted: {bronze_count}")

if bronze_count == 0:
    raise Exception(f"No PDF files matched expected path pattern under {scan_path}.")

raw_df = spark.createDataFrame(rows)
raw_df.show(3, truncate=False)

In [ ]:
# Cell 5 — Cast columns and build derived fields
# raw_df has: invoice_id(str), year(str), month(str), day(str), file_size_kb(float), bronze_path(str)

enriched_df = (
    raw_df
    .withColumn("invoice_id",    F.trim(F.col("invoice_id").cast("string")))
    .withColumn("invoice_year",  F.col("year").cast("integer"))
    .withColumn("invoice_month", F.col("month").cast("integer"))
    .withColumn("invoice_day",   F.col("day").cast("integer"))
    .withColumn("file_size_kb",  F.col("file_size_kb").cast("decimal(10,2)"))
    .withColumn("bronze_path",   F.trim(F.col("bronze_path").cast("string")))
    .drop("year", "month", "day")
    .withColumn(
        "invoice_date",
        F.to_date(
            F.concat_ws("-",
                F.col("invoice_year").cast("string"),
                F.lpad(F.col("invoice_month").cast("string"), 2, "0"),
                F.lpad(F.col("invoice_day").cast("string"),   2, "0"),
            ), "yyyy-MM-dd"
        )
    )
    .withColumn(
        "invoice_number",
        F.regexp_extract(F.col("invoice_id"), r"(\d+)$", 1).cast("long")
    )
)

print("After enrichment and cast:")
enriched_df.printSchema()
enriched_df.show(3, truncate=False)

In [ ]:
# Cell 6 — Drop rows with NULL invoice_id or NULL invoice_date

def write_quarantine(df, reason):
    if df.limit(1).count() == 0:
        return
    (
        df.withColumn("reject_reason", F.lit(reason))
          .withColumn("reject_ts",     F.lit(RUN_TS).cast("timestamp"))
          .write.format("delta").mode("append").option("mergeSchema", "true")
          .save(QUARANTINE_PATH)
    )

null_id_df   = enriched_df.filter(F.col("invoice_id").isNull() | (F.col("invoice_id") == ""))
null_date_df = enriched_df.filter(F.col("invoice_id").isNotNull() & F.col("invoice_date").isNull())

write_quarantine(null_id_df,   "null_invoice_id")
write_quarantine(null_date_df, "null_invoice_date")

clean_df = (
    enriched_df
    .filter(F.col("invoice_id").isNotNull() & (F.col("invoice_id") != ""))
    .filter(F.col("invoice_date").isNotNull())
)

print(f"Null invoice_id dropped  : {null_id_df.count()}")
print(f"Null invoice_date dropped: {null_date_df.count()}")
print(f"Clean rows               : {clean_df.count()}")

In [ ]:
# Cell 7 — Add Silver audit columns + deduplicate on invoice_id

audited_df = (
    clean_df
    .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
    .withColumn("silver_load_type",   F.lit(LOAD_TYPE))
    .withColumn("silver_pipeline",    F.lit(PIPELINE_NAME))
)

# Deduplicate — keep latest record per invoice_id
window = Window.partitionBy("invoice_id").orderBy(F.col("silver_ingested_at").desc())
deduped_df = (
    audited_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

print(f"After dedup: {deduped_df.count()} rows")

In [ ]:
# Cell 8 — Write to Silver Delta (overwrite or MERGE)

if LOAD_TYPE == "full" or not folder_exists_dbfs(SILVER_PATH):
    (
        deduped_df.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .save(SILVER_PATH)
    )
    print(f"Full overwrite complete — {deduped_df.count()} rows written")
else:
    delta_table = DeltaTable.forPath(spark, SILVER_PATH)
    (
        delta_table.alias("target")
        .merge(deduped_df.alias("source"), "target.invoice_id = source.invoice_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE complete — {deduped_df.count()} rows upserted")

In [ ]:
# Cell 9 — Verify

silver_df = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver invoice rows : {silver_df.count()}")
silver_df.printSchema()
silver_df.show(5, truncate=False)

print("\nInvoices per month:")
silver_df.groupBy("invoice_year", "invoice_month").count().orderBy("invoice_year", "invoice_month").show()